In [24]:
import pandas as pd
df = pd.read_csv("spam_ham.csv")
df.head(5)

,label,email_text
0,ham,Your subscription has been successfully renewe...
1,ham,Reminder: Team meeting at 3 PM today. RefID-1
2,ham,Let's schedule a meeting for tomorrow at 10 AM...
3,ham,Can you review the project proposal and share ...
4,ham,Your subscription has been successfully renewe...


In [25]:
print(df.shape)

(2000, 2)


In [26]:
print(df['label'].value_counts())

label
ham     1000
spam    1000
Name: count, dtype: int64


In [27]:
df["out"] = df["label"].apply(lambda x: 1 if x=="spam" else 0)

# Convert labels to numeric
 #df['label'] = df['label'].map({'ham': 0, 'spam': 1})

# Verify conversion
#print(df.head())
#print(df['label'].value_counts())

In [28]:
df.head(-5)

,label,email_text,out
0,ham,Your subscription has been successfully renewe...,0
1,ham,Reminder: Team meeting at 3 PM today. RefID-1,0
2,ham,Let's schedule a meeting for tomorrow at 10 AM...,0
3,ham,Can you review the project proposal and share ...,0
4,ham,Your subscription has been successfully renewe...,0
...,...,...,...
1990,spam,Earn $5000 per week working from home! PromoCo...,1
1991,spam,Limited time offer! Get 90% discount now! Prom...,1
1992,spam,Winner! Claim your free vacation package today...,1
1993,spam,Limited time offer! Get 90% discount now! Prom...,1


In [29]:
from sklearn.model_selection import train_test_split
X = df['email_text']
y = df['out']

X_train, X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [30]:
print("training size:",len(X_train))
print("testing size:", len(X_test))

training size: 1600
testing size: 400


In [31]:
from transformers.models.bert import BertTokenizer

In [32]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
sample_text  = X_train.iloc[0]
encoding = tokenizer(
    sample_text,
    padding = "max_length",
    truncation = True,
    max_length = 16
)
print("original text:",sample_text)
print("Input ids:", encoding['input_ids'])
print("Attention Mask:", encoding['attention_mask'])

original text: Hi John, please find the attached report for last quarter. RefID-968
Input ids: [101, 7632, 2198, 1010, 3531, 2424, 1996, 4987, 3189, 2005, 2197, 4284, 1012, 25416, 3593, 102]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [33]:
train_encode = tokenizer(
    list(X_train),
    padding = True,
    truncation = True,
    max_length = 128
)
test_encode = tokenizer(
    list(X_test),
    padding = True,
    truncation = True,
    max_length = 128
)

print("Train sample toknize:", len(train_encode['input_ids']))
 
 #2️⃣ padding=True
#Means:
#Pad all sentences to same length within batch


Train sample toknize: 1600


This code creates a custom PyTorch Dataset class that stores tokenized email data (like input_ids, attention_mask) and their corresponding labels (spam / not spam).

It allows the model to easily fetch one email sample at a time in tensor format during training, and also tells how many total samples are present.

In [34]:
import torch
class EmailDataset(torch.utils.data.Dataset):
    def __init__(self,encodings,lables):
        self.encodings = encodings
        self.lables = lables.reset_index(drop=True)

    def __getitem__ (self,idx):
        item = {key : torch.tensor(val[idx])
                for key,val in self.encodings.items()}
        item['labels'] = torch.tensor(self.lables[idx])  
        return item

    def __len__ (self):
        return len(self.lables)  

In [35]:
trin_dataset = EmailDataset(train_encode,y_train)
test_dataset = EmailDataset(test_encode, y_test)

In [36]:
print(trin_dataset[0])

{'input_ids': tensor([  101,  7632,  2198,  1010,  3531,  2424,  1996,  4987,  3189,  2005,
         2197,  4284,  1012, 25416,  3593,  1011,  5986,  2620,   102,     0,
            0]), 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]), 'labels': tensor(0)}


In [37]:
print(torch.__version__)
print(torch.cuda.is_available())

2.10.0+cpu
False


In [38]:
from transformers import BertModel

model = BertModel.from_pretrained("bert-base-uncased")
print("Base model loaded successfully")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Base model loaded successfully


It is a Hugging Face Transformers class used to load a pre-trained model (like BERT) that is specially designed for text classification tasks.

In [39]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

print("Classification model loaded successfully")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Classification model loaded successfully


TrainingArguments is used to configure the training process such as epochs, batch size, learning rate, logging and checkpoint saving

In [40]:
from transformers import TrainingArguments

In [41]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",   # ✅ correct for v5
    logging_dir="./logs",
    logging_steps=10
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [42]:
from transformers import Trainer
trainer = Trainer(
    model= model,
    args = training_args,
    train_dataset= trin_dataset,
    eval_dataset= test_dataset
)

In [43]:
trainer.train()

c:\Users\adars\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(trin_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

print("DataLoaders ready") 

DataLoaders ready


In [48]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print("Test Accuracy:", accuracy)

Test Accuracy: 1.0


In [49]:
from sklearn.metrics import classification_report, confusion_matrix

all_preds = []
all_labels = []

model.eval()

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [50]:
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[199   0]
 [  0 201]]


In [51]:
model.save_pretrained("./saved_model")
tokenizer.save_pretrained("./saved_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_model\\tokenizer_config.json', './saved_model\\tokenizer.json')